# Deploy Order Agent to Amazon Bedrock AgentCore Runtime

This tutorial series builds a **5-agent e-commerce assistant** using **Strands Agents GraphBuilder** for deterministic routing across Amazon Bedrock AgentCore runtimes. In this notebook, you deploy the Order Agent -- an A2A server that queries customer orders from Amazon DynamoDB using MCP tools.

**Notebook 3 of 5** -- The Graph Orchestrator routes to this agent for ORDER and RECOMMEND intents.

## Architecture Overview

This tutorial deploys 5 agents across 5 Amazon Bedrock AgentCore runtimes. The Order Agent (highlighted below) handles order queries:

| Runtime | Agent | Protocol | Tools |
|---------|-------|----------|-------|
| 1 | Classifier | A2A (port 9000) | None (pure LLM) |
| 2 | Product | A2A (port 9000) | HTTP API tools |
| **3** | **Order** | **A2A (port 9000)** | **DynamoDB via MCP** |
| 4 | Recommendation | A2A (port 9000) | None (LLM synthesis) |
| 5 | Graph Orchestrator | HTTP (port 8080) | GraphBuilder + A2A clients |

The Order Agent is invoked for:
- **ORDER** intent: Direct order lookup from classifier
- **RECOMMEND** intent: Provides order history to Recommendation Agent (runs in parallel with Product Agent)

**MCP Integration:** The agent uses `awslabs.dynamodb-mcp-server` via stdio transport, launched as a subprocess with timeout-protected initialization.

## Prerequisites

- [AWS CLI](https://aws.amazon.com/cli/) installed and configured
- Python 3.10 or higher
- Docker or Podman installed (only for local container builds; not required if using CodeBuild)
- Claude Sonnet 4 model access in Amazon Bedrock
- **Notebook 01 completed**: Classifier Agent deployed
- **Notebook 02 completed**: Product Agent deployed

In [ ]:
import json
import os
from pathlib import Path
from urllib.parse import quote
from uuid import uuid4

import boto3

NOTEBOOK_DIR = Path.cwd()

from utils import (
    SSM_ORDER_AGENT_URL,
    SSM_ORDERS_TABLE,
    DYNAMODB_TABLE_NAME,
    ORDER_AGENT_NAME,
    ORDER_ROLE_NAME,
    create_agentcore_role,
    store_agent_url,
    create_orders_table,
    seed_orders,
    delete_orders_table,
)

session = boto3.Session()
region = session.region_name or "us-west-2"
account_id = boto3.client("sts").get_caller_identity()["Account"]

print(f"Region: {region}")
print(f"Account: {account_id}")
print(f"Agent Name: {ORDER_AGENT_NAME}")

---
## Step 1: Create Order Agent with MCP Integration

The Order Agent uses the Strands Agents `MCPClient` with stdio transport to run `awslabs.dynamodb-mcp-server` as a subprocess. This provides DynamoDB query capabilities without writing custom tool code.

**MCP Integration Architecture:**
```
A2A Request -> Order Agent -> MCPClient (stdio) -> dynamodb-mcp-server -> DynamoDB
```

### Set Up DynamoDB Orders Table

Create the DynamoDB table and seed it with sample order data before deploying the agent.

In [ ]:
print("Creating DynamoDB orders table...")
result = create_orders_table(table_name=DYNAMODB_TABLE_NAME, region=region)
table_name = DYNAMODB_TABLE_NAME
print(f"Table: {table_name} ({result['status']})")

In [ ]:
orders_file = NOTEBOOK_DIR / "sample_data" / "orders.json"
print(f"Loading sample orders from: {orders_file}")
result = seed_orders(table_name=table_name, json_file=orders_file, region=region)
print(f"Seeded {result['count']} orders")

In [ ]:
result = store_agent_url(
    param_name=SSM_ORDERS_TABLE,
    url=table_name,
    region=region,
)
print(result["message"])
print(f"Parameter version: {result['version']}")

### Code Structure

The following cell creates `order_agent/a2a_server.py`:

| Section | What It Does |
|---------|--------------|
| Configuration | Constants for port (9000), SSM parameter paths, MCP timeouts, and system prompt |
| `ToolLoggingHandler` | Callback that logs tool invocations to CloudWatch for debugging |
| `server_lifespan` | Async context manager that initializes MCP subprocess after port binding |
| Agent Creation | Agent created with `tools=[]`, MCP tools attached during lifespan |
| FastAPI + A2A | Health check at `/ping`, A2A server mounted at root |

In [ ]:
%%writefile order_agent/a2a_server.py
"""Order Agent deployed to Amazon Bedrock AgentCore with A2A protocol support.

Queries customer orders from Amazon DynamoDB using the awslabs.dynamodb-mcp-server
MCP tool. Uses async lifespan pattern for MCP subprocess initialization to ensure
health checks pass before tool loading begins.

Key concepts demonstrated:
- A2A protocol server for graph-based multi-agent orchestration (port 9000)
- MCP client with stdio transport for DynamoDB access
- Two-phase initialization: fast startup, then async tool loading
- Graceful degradation if MCP subprocess fails
"""

import asyncio
import json
import logging
import os
from contextlib import asynccontextmanager
from typing import Any

import boto3
import uvicorn
from fastapi import FastAPI
from mcp import StdioServerParameters, stdio_client
from strands import Agent
from strands.models import BedrockModel
from strands.multiagent.a2a import A2AServer
from strands.telemetry import StrandsTelemetry
from strands.tools.mcp import MCPClient

# --- Logging ---

logging.basicConfig(level=logging.INFO)
logging.getLogger("strands").setLevel(logging.INFO)
logging.getLogger("a2a").setLevel(logging.DEBUG)
logging.getLogger("strands.multiagent.a2a").setLevel(logging.DEBUG)
logger = logging.getLogger(__name__)

# Enable distributed tracing with AWS X-Ray
StrandsTelemetry().setup_otlp_exporter()

# --- Configuration ---

PORT = 9000  # A2A protocol uses port 9000 (HTTP protocol uses 8080)
SSM_ORDER_AGENT_URL = "/ecommerce-graph/order-agent-url"
SSM_ORDERS_TABLE = "/ecommerce-graph/orders-table"

# Timeouts prevent container hangs if MCP server fails to start
MCP_STARTUP_TIMEOUT = 10.0  # Max seconds to wait for subprocess
MCP_TOOLS_TIMEOUT = 5.0     # Max seconds to fetch tool definitions

SYSTEM_PROMPT_TEMPLATE = """You are an Order Agent for an e-commerce shopping assistant.

Query the "{table_name}" DynamoDB table for order information.
Return order details including product_ids so downstream agents can use this data."""


# --- Callback Handler ---


class ToolLoggingHandler:
    """Log tool invocations to CloudWatch for debugging multi-agent workflows."""

    def __init__(self) -> None:
        self.logged_tool_ids: set[str] = set()
        self.tool_count = 0

    def __call__(self, **kwargs: Any) -> None:
        message = kwargs.get("message", {})
        if isinstance(message, dict) and message.get("role") == "assistant":
            for content in message.get("content", []):
                if isinstance(content, dict):
                    tool_use = content.get("toolUse")
                    if tool_use:
                        tool_id = tool_use.get("toolUseId")
                        if tool_id and tool_id not in self.logged_tool_ids:
                            self.logged_tool_ids.add(tool_id)
                            self.tool_count += 1
                            tool_name = tool_use.get("name", "Unknown")
                            tool_input = tool_use.get("input", {})
                            logger.info(f"=== TOOL #{self.tool_count}: {tool_name} ===")
                            logger.info(f"TOOL INPUT: {json.dumps(tool_input)}")

        if kwargs.get("complete") and kwargs.get("data"):
            logger.info(f"=== COMPLETE: {len(kwargs.get('data', ''))} chars ===")


# --- Helper Functions ---


def get_table_name() -> str:
    """Get DynamoDB table name from environment variable or SSM Parameter Store."""
    if table_name := os.environ.get("ORDERS_TABLE"):
        logger.info(f"Using table from env: {table_name}")
        return table_name

    try:
        ssm = boto3.client("ssm")
        response = ssm.get_parameter(Name=SSM_ORDERS_TABLE, WithDecryption=True)
        table_name = response["Parameter"]["Value"]
        logger.info(f"Using table from SSM: {table_name}")
        return table_name
    except Exception as e:
        logger.error(f"Could not get table name: {e}")
        raise


def get_runtime_url() -> str | None:
    """Get AgentCore runtime URL for agent card generation."""
    if url := os.environ.get("ORDER_AGENT_URL"):
        logger.info(f"Using runtime URL from env: {url}")
        return url

    try:
        ssm = boto3.client("ssm")
        response = ssm.get_parameter(Name=SSM_ORDER_AGENT_URL, WithDecryption=True)
        url = response["Parameter"]["Value"]
        logger.info(f"Using runtime URL from SSM: {url}")
        return url
    except Exception as e:
        logger.warning(f"Could not get runtime URL: {e}")
        return None


# --- Agent Creation ---
# Create agent with tools=[] for fast module import. MCP tools are added later
# in server_lifespan() after the server binds to port 9000.

region = boto3.session.Session().region_name or "us-west-2"

order_agent = Agent(
    name="Ecommerce_Graph_Order",
    description="Looks up order information from DynamoDB for customers",
    system_prompt="Order agent initializing...",
    model=BedrockModel(
        model_id="us.anthropic.claude-sonnet-4-20250514-v1:0",
        region_name=region,
    ),
    tools=[],
    callback_handler=ToolLoggingHandler(),
)
logger.info(f"Agent created: {order_agent.name}")

# MCP connection handle - initialized in server_lifespan, cleaned up on shutdown
mcp_tool_connection: MCPClient | None = None


# --- Server Lifespan (startup and shutdown) ---


@asynccontextmanager
async def server_lifespan(app: FastAPI):
    """Initialize MCP tools after server starts, clean up on shutdown.

    This runs AFTER uvicorn binds to port 9000, so health checks pass immediately.
    The MCP subprocess starts here, not during module import.

    Sequence:
    1. Module imports -> agent created with tools=[] (fast)
    2. uvicorn binds to port 9000 -> /ping endpoint ready
    3. AgentCore health checks pass -> container marked healthy
    4. THIS FUNCTION runs -> MCP subprocess starts -> tools added to agent
    5. Agent ready to handle A2A requests with DynamoDB access
    """
    global mcp_tool_connection

    logger.info("=== Order Agent Startup ===")

    try:
        table_name = get_table_name()

        # Launch DynamoDB MCP server as subprocess via uvx
        mcp_tool_connection = MCPClient(
            lambda: stdio_client(
                StdioServerParameters(
                    command="uvx",
                    args=["awslabs.dynamodb-mcp-server@latest"],
                    env={
                        **os.environ,
                        "AWS_REGION": region,
                        "DDB_TABLE_NAME": table_name,
                    },
                )
            )
        )

        # Start MCP subprocess with timeout protection
        event_loop = asyncio.get_event_loop()
        start_mcp_subprocess = mcp_tool_connection.__enter__
        await asyncio.wait_for(
            event_loop.run_in_executor(None, start_mcp_subprocess),
            timeout=MCP_STARTUP_TIMEOUT,
        )
        logger.info("MCP tool connection started")

        # Fetch available tools from MCP server
        mcp_tools = await asyncio.wait_for(
            event_loop.run_in_executor(None, mcp_tool_connection.list_tools_sync),
            timeout=MCP_TOOLS_TIMEOUT,
        )
        logger.info(f"Loaded {len(mcp_tools)} tools from MCP server")

        # Attach tools to agent
        order_agent.tools = mcp_tools
        order_agent.system_prompt = SYSTEM_PROMPT_TEMPLATE.format(table_name=table_name)

        logger.info(f"=== Startup Complete: {order_agent.name} with {len(mcp_tools)} tools ===")

    except asyncio.TimeoutError:
        logger.error("MCP initialization timed out - agent will operate without tools")
    except Exception as e:
        logger.error(f"Startup error: {e} - agent will operate without tools")

    yield

    # Shutdown: clean up MCP subprocess
    logger.info("=== Shutting Down ===")
    if mcp_tool_connection:
        try:
            stop_mcp_subprocess = mcp_tool_connection.__exit__
            stop_mcp_subprocess(None, None, None)
            logger.info("MCP tool connection closed")
        except Exception as e:
            logger.error(f"Error closing MCP: {e}")


# --- FastAPI App and A2A Server ---

runtime_url = get_runtime_url()

a2a_server = A2AServer(
    agent=order_agent,
    host="0.0.0.0",
    port=PORT,
    http_url=runtime_url,
    serve_at_root=True,
)

app = FastAPI(title="Order Agent A2A Server", lifespan=server_lifespan)


@app.get("/ping")
def ping():
    """Health check endpoint for AgentCore container probes."""
    return {"status": "healthy"}


app.mount("/", a2a_server.to_fastapi_app())

if __name__ == "__main__":
    uvicorn.run(app, host="0.0.0.0", port=PORT)

In [ ]:
%%writefile order_agent/requirements.txt
strands-agents[a2a,otel]
strands-agents-tools
fastapi
uvicorn
boto3
mcp

---
## Step 2: Deploy to Amazon Bedrock AgentCore

The `bedrock-agentcore-starter-toolkit` handles the deployment pipeline:

1. **Creates IAM role** -- Grants permissions for ECR, Bedrock, DynamoDB access, and CloudWatch logging
2. **Configures runtime** -- Packages agent code into a deployable container
3. **Launches runtime** -- Builds image, pushes to ECR, creates AgentCore runtime

### Create IAM Role and Configure Runtime

The Order Agent needs additional DynamoDB permissions beyond the base AgentCore role.

| Parameter | Value | Purpose |
|-----------|-------|---------|
| `entrypoint` | `a2a_server.py` | Python file that starts the A2A server |
| `protocol` | `A2A` | Enables Agent-to-Agent communication on port 9000 |
| `agent_name` | `ecommerce_graph_order` | Unique identifier for this runtime |

In [ ]:
table_arn = f"arn:aws:dynamodb:{region}:{account_id}:table/{table_name}"
index_arn = f"{table_arn}/index/*"

dynamodb_permissions = [
    {
        "Effect": "Allow",
        "Action": [
            "dynamodb:GetItem",
            "dynamodb:Query",
            "dynamodb:Scan",
            "dynamodb:DescribeTable",
            "dynamodb:BatchGetItem",
        ],
        "Resource": [table_arn, index_arn],
    }
]

order_role_arn = create_agentcore_role(
    ORDER_ROLE_NAME, account_id, region, extra_permissions=dynamodb_permissions
)
print(f"IAM Role ARN: {order_role_arn}")

In [ ]:
from bedrock_agentcore_starter_toolkit import Runtime

order_agent_dir = NOTEBOOK_DIR / "order_agent"
os.chdir(order_agent_dir)

order_runtime = Runtime()
order_runtime.configure(
    entrypoint="a2a_server.py",
    execution_role=order_role_arn,
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=region,
    agent_name=ORDER_AGENT_NAME,
    protocol="A2A",
)

os.chdir(NOTEBOOK_DIR)
print(f"Runtime configured: {ORDER_AGENT_NAME}")

### Fix Dockerfile Permissions

AgentCore containers run as the `bedrock_agentcore` user, not root. The auto-generated Dockerfile uses `COPY . .` which preserves host file ownership, causing permission errors. This cell updates it to `COPY --chown=bedrock_agentcore:bedrock_agentcore . .`.

In [ ]:
# dockerfile_path = order_agent_dir / "Dockerfile"
# if dockerfile_path.exists():
#     content = dockerfile_path.read_text()
#     if "COPY . ." in content and "--chown" not in content:
#         content = content.replace("COPY . .", "COPY --chown=bedrock_agentcore:bedrock_agentcore . .")
#         dockerfile_path.write_text(content)
#         print("Dockerfile updated with correct ownership")
#     else:
#         print("Dockerfile already has correct ownership or uses different COPY syntax")
# else:
#     print("Dockerfile not found - it will be generated during launch")

### Launch Agent

`Runtime.launch()` builds the Docker image, pushes it to ECR, and creates the AgentCore runtime.

**Note:** First deployment takes 5-10 minutes. Order Agent may take slightly longer due to MCP server dependencies.

In [ ]:
print("Launching Order Agent (this may take several minutes)...")
os.chdir(order_agent_dir)
order_launch = order_runtime.launch(auto_update_on_conflict=True)
print(f"Order Agent ARN: {order_launch.agent_arn}")
ORDER_AGENT_ARN = order_launch.agent_arn
os.chdir(NOTEBOOK_DIR)

### Get Runtime URL

Once the runtime reaches `ACTIVE` or `READY` status, construct the invocation URL from the runtime ARN.

In [ ]:
os.chdir(order_agent_dir)
status_response = order_runtime.status()
status = status_response.endpoint.get("status", "")
order_agent_url = None

print(f"Order Agent Status: {status}")

if status.upper() in ["ACTIVE", "READY"]:
    agent_runtime_arn = status_response.endpoint.get("agentRuntimeArn")
    escaped_arn = quote(agent_runtime_arn, safe="")
    order_agent_url = f"https://bedrock-agentcore.{region}.amazonaws.com/runtimes/{escaped_arn}/invocations"
    print(f"Runtime URL: {order_agent_url}")
else:
    print(f"Agent not ready. Current status: {status}")

os.chdir(NOTEBOOK_DIR)

### Store Runtime URL in Parameter Store

Store the URL in AWS Systems Manager Parameter Store so other components can discover this agent:
- **Agent container** reads it at startup to populate the agent card's `http_url` field
- **Graph Orchestrator** (Notebook 5) retrieves it to configure `A2AClientToolProvider` for the order node

In [ ]:
if status.upper() in ["ACTIVE", "READY"]:
    result = store_agent_url(
        param_name=SSM_ORDER_AGENT_URL,
        url=order_agent_url,
        region=region,
    )
    print(result["message"])
    print(f"Parameter version: {result['version']}")
else:
    print("Skipping SSM storage - agent not ready")

In [ ]:
print("=" * 60)
print("Order Agent Deployment Summary")
print("=" * 60)
print(f"Agent Name: {ORDER_AGENT_NAME}")
print(f"Agent ARN: {ORDER_AGENT_ARN}")
print(f"IAM Role: {order_role_arn}")
print(f"Runtime URL: {order_agent_url or 'Not available - agent not ready'}")
print(f"DynamoDB Table: {table_name}")
print(f"SSM Parameters:")
print(f"  Agent URL: {SSM_ORDER_AGENT_URL}")
print(f"  Table Name: {SSM_ORDERS_TABLE}")
print("=" * 60)

---
## Step 3: Test Order Agent

Verify the agent works standalone before integrating with the Graph Orchestrator.

**Example queries to try:**
- `"What orders does customer CUST-101 have?"` -- customer order lookup
- `"Show me recent orders"` -- general order query
- `"What did CUST-102 purchase?"` -- specific customer history

In [ ]:
test_message = "What orders does customer CUST-101 have?"

payload = {
    "jsonrpc": "2.0",
    "method": "message/send",
    "id": str(uuid4()),
    "params": {
        "message": {
            "messageId": str(uuid4()),
            "role": "user",
            "parts": [{"kind": "text", "text": test_message}],
        }
    },
}

os.chdir(order_agent_dir)
print(f"Testing: '{test_message}'")
print("-" * 40)
response = order_runtime.invoke(payload, session_id=str(uuid4()))
print(f"\nResponse:\n{response}")
os.chdir(NOTEBOOK_DIR)

### Verify Agent Card (A2A Discovery)

The `A2AServer` wrapper automatically generates an A2A-compliant agent card from your Strands agent.

In [ ]:
from botocore.auth import SigV4Auth
from botocore.awsrequest import AWSRequest
import requests

agent_card_url = f"{order_agent_url}/.well-known/agent-card.json"
credentials = boto3.Session().get_credentials()
request = AWSRequest(method="GET", url=agent_card_url)
SigV4Auth(credentials, "bedrock-agentcore", region).add_auth(request)

response = requests.get(
    agent_card_url,
    headers=dict(request.headers),
)
print(f"Status: {response.status_code}")
if response.status_code == 200:
    card = response.json()
    print(json.dumps(card, indent=2))

---
## Next Steps

| Notebook | What You'll Build |
|----------|-------------------|
| **4. Deploy Recommendation Agent** | LLM synthesis agent for personalized recommendations |
| **5. Deploy Graph Orchestrator** | GraphBuilder DAG that routes through all agents with conditional edges |

---
## Cleanup (Optional)

Run this section to delete all resources created by this notebook.

In [ ]:
print("Destroying Order Agent...")
os.chdir(order_agent_dir)
try:
    order_runtime.destroy(delete_ecr_repo=True)
    print("Order Agent destroyed")
except Exception as e:
    print(f"Error: {e}")
os.chdir(NOTEBOOK_DIR)

In [ ]:
print("Deleting DynamoDB table...")
try:
    result = delete_orders_table(table_name=DYNAMODB_TABLE_NAME, region=region)
    print(f"Status: {result['status']}")
    print(f"Message: {result['message']}")
except Exception as e:
    print(f"Error deleting DynamoDB table: {e}")

In [ ]:
ssm = boto3.client("ssm", region_name=region)
for param in [SSM_ORDER_AGENT_URL, SSM_ORDERS_TABLE]:
    try:
        ssm.delete_parameter(Name=param)
        print(f"Deleted SSM parameter: {param}")
    except Exception as e:
        print(f"Error deleting {param}: {e}")

print("Cleaning up auto-generated files...")
for cleanup_file in ["Dockerfile", ".dockerignore"]:
    cleanup_path = order_agent_dir / cleanup_file
    if cleanup_path.exists():
        cleanup_path.unlink()
        print(f"  Deleted: {cleanup_file}")